# Decompondo a Variabilidade da Liquidação de Sinistros na Hierarquia Organizacional de uma Seguradora com PROC NESTED

## Resumo Executivo

Uma seguradora de ramos elementares (danos e responsabilidade civil) quer saber
**onde** se origina a inconsistência nas liquidações de sinistros: ela é
causada por diferenças entre regiões geográficas, entre filiais, entre
reguladores de sinistros individuais, ou simplesmente por ruído de
sinistro para sinistro? Este notebook constrói um livro sintético e
totalmente aninhado de sinistros de auto (Região > Filial > Regulador >
sinistro) e usa **PROC NESTED** para realizar uma análise de variância de
efeitos aleatórios, estimando o componente de variância contribuído por
cada nível da hierarquia e reportando cada um como um percentual do total.
O resultado diz à administração qual camada organizacional deve ser o
alvo para melhorias na consistência das liquidações.


## Fontes de Dados

Todos os dados são gerados internamente pelo primeiro DATA step — sem
arquivos externos, sem rede. O desenho é **completamente aninhado**: cada
filial pertence a exatamente uma região, cada regulador a exatamente uma
filial, e cada sinistro a exatamente um regulador.

| Conjunto de Dados | Linhas | Variável | Tipo | Papel | Descrição |
|---------|------|----------|------|------|-------------|
| `claims` | 40 | `Region` | char | CLASS (nível 1, mais externo) | Região geográfica (5 regiões: NE, SE, MO, CO, SO) |
| | | `Branch` | char | CLASS (nível 2, aninhado em Region) | Código da filial (2 filiais por região) |
| | | `Adjuster` | char | CLASS (nível 3, aninhado em Branch) | ID do regulador de sinistros (2 reguladores por filial) |
| | | `Settlement` | num | VAR (dependente) | Indenização paga no sinistro de auto, em USD |
| | | `CycleDays` | num | VAR (dependente) | Dias entre o primeiro aviso de sinistro e a liquidação |

Estrutura sintética: 5 regiões x 2 filiais x 2 reguladores x 2 sinistros =
40 observações. Um efeito aleatório de região, um efeito aleatório de
filial dentro da região, um efeito aleatório de regulador dentro da
filial, e um resíduo no nível do sinistro são sobrepostos aditivamente
via `rand('NORMAL', ...)` para que os componentes de variância sejam
recuperáveis. O efeito de região é sorteado com o maior desvio-padrão
(2.200) para que *onde* um sinistro é tratado seja o fator dominante.
Semente fixada com `call streaminit(20260531)`. Um desenho compacto e
totalmente balanceado mantém todos os níveis da hierarquia povoados com
graus de liberdade reais.


# Decompondo a Variabilidade da Liquidação de Sinistros com PROC NESTED

Quando uma seguradora de ramos elementares analisa quanto paga para
liquidar sinistros de auto, frequentemente encontra ampla variação. Parte
dessa variação é inevitável (todo acidente é diferente), mas parte dela
reflete fatores **organizacionais** — uma região opera de forma mais
generosa que outra, uma filial é mais generosa, um regulador individual é
um outlier.

Os dados têm uma estrutura estritamente **aninhada** (hierárquica):

```
Region  ->  Branch (aninhada em Region)  ->  Adjuster (aninhado em Branch)  ->  sinistros individuais
```

Uma filial pertence a exatamente uma região e um regulador a exatamente
uma filial, então os fatores são aninhados em vez de cruzados. `PROC
NESTED` realiza uma análise de variância de efeitos aleatórios exatamente
para esse desenho e estima um **componente de variância** em cada nível,
expresso como um percentual do total. Esse detalhamento percentual
responde à pergunta de negócio: *qual camada da organização devemos
visar para tornar as liquidações mais consistentes?*


## Etapa 1 — Gerar um livro sintético e totalmente aninhado de sinistros

Simulamos 5 regiões, cada uma com 2 filiais, cada uma com 2 reguladores,
cada um tratando 2 sinistros (40 linhas no total). Construímos a resposta
a partir de efeitos aleatórios aditivos para que cada nível genuinamente
contribua com variância:

- um efeito de **região** (maior dispersão — as regiões diferem mais),
- um efeito de **filial dentro da região**,
- um efeito de **regulador dentro da filial**,
- e um **resíduo no nível do sinistro** (o ruído dentro do regulador).

`call streaminit` fixa a semente para reprodutibilidade; cada efeito é
sorteado com `rand('NORMAL', média, dp)`. O efeito de região usa o maior
desvio-padrão, então deve reivindicar a maior fatia da variância total.
Também geramos uma segunda resposta, `CycleDays`, que compartilha a mesma
hierarquia para que possamos demonstrar a análise multirresposta mais
adiante.


In [1]:
DADOS claims;
   CHAMAR streaminit(20260531);
   COMPRIMENTO Region $2 Branch $6 Adjuster $9;

   /* Efeito aleatório de nível de Região: as regiões diferem mais. */
   FAZER r = 1 ATÉ 5;
      Region = SCAN('NE SE MO CO SO', r);
      RegionEffect  = rand('NORMAL', 0, 2200);
      RegionCycle   = rand('NORMAL', 0, 11);

      /* Filial aninhada dentro da região. */
      FAZER b = 1 ATÉ 2;
         Branch = catx('-', Region, PUT(b, z2.));
         BranchEffect = rand('NORMAL', 0, 700);
         BranchCycle  = rand('NORMAL', 0, 4);

         /* Regulador aninhado dentro da filial. */
         FAZER a = 1 ATÉ 2;
            Adjuster = catx('-', Branch, PUT(a, z1.));
            AdjEffect = rand('NORMAL', 0, 450);
            AdjCycle  = rand('NORMAL', 0, 2.5);

            /* Sinistros individuais tratados por este regulador. */
            FAZER claim = 1 ATÉ 2;
               Settlement = 8500
                          + RegionEffect + BranchEffect + AdjEffect
                          + rand('NORMAL', 0, 1100);
               CycleDays  = 21
                          + RegionCycle + BranchCycle + AdjCycle
                          + rand('NORMAL', 0, 6);
               SE Settlement < 0 ENTÃO Settlement = 0;
               SE CycleDays  < 1 ENTÃO CycleDays  = 1;
               SAÍDA;
            FIM;
         FIM;
      FIM;
   FIM;

   MANTER Region Branch Adjuster Settlement CycleDays;
EXECUTAR;



NOTE: DATA claims


NOTE: Wrote claims (40 rows, 5 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds


## Etapa 2 — Ordenar pela hierarquia de aninhamento

`PROC NESTED` exige que os dados de entrada estejam **ordenados pelas
variáveis CLASS na ordem em que serão listadas** — o fator mais externo
primeiro. Ordenamos por `Region Branch Adjuster` para que o procedimento
possa percorrer a hierarquia corretamente.


In [2]:
PROCEDIMENTO SORT DADOS=claims;
   POR Region Branch Adjuster;
EXECUTAR;



NOTE: PROC SORT data=claims

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 40 rows from claims.
NOTE: Wrote claims (40 rows, 5 columns).
NOTE: PROC SORT statement used.


## Etapa 3 — Análise de componentes de variância do valor da liquidação

A análise central. Listamos as variáveis CLASS da mais externa para a
mais interna (`Region Branch Adjuster`); a replicação mais interna — os
sinistros individuais — é tratada automaticamente como o termo de erro.
`VAR Settlement` nomeia a única resposta.

A instrução `LABEL` fornece um nome de exibição mais amigável para a
resposta no banner de saída. A saída produz os coeficientes dos
quadrados médios esperados, a tabela de análise de variância (com um
teste F de cada componente de variância contra zero), e a estimativa do
**componente de variância** em cada nível, junto com seu **percentual do
total**.


In [3]:
TÍTULO 'Componentes de Variância das Liquidações de Sinistros de Auto';
title2 'ANOVA de Efeitos Aleatórios: Região / Filial / Regulador';

PROCEDIMENTO nested DADOS=claims;
   CLASSE Region Branch Adjuster;
   VARIÁVEL Settlement;
   RÓTULO Region='Região' Branch='Filial' Adjuster='Regulador de Sinistros'
         Settlement = 'Indenização Paga (USD)';
EXECUTAR;


                             Componentes de Variância das Liquidações de Sinistros de Auto                              
                                ANOVA de Efeitos Aleatórios: Região / Filial / Regulador                                


                       Nested Random Effects Analysis of Variance

Nested Random Effects Analysis of Variance for Variable Indenização Paga (USD)
Variance Source       DF  Sum of Squares       F Value      Pr > F  Error Term   Mean Square  Variance Component    Percent of Total    EMS Coef
Região                 4  62114122.126866          6.71      0.0304    Filial  15528530.531717  1651824.844989             54.4057      8.0000
Filial                 5  11569658.859028          1.89      0.1829  Regulador de Sinistros  2313931.771806   272682.828942              8.9813      4.0000
Regulador de Sinistros  10  12232004.560376          1.22      0.3348     Error  1223200.456038   111582.782346              3.6752      2.0000
Error                 


NOTE: Option TITLE changed to Componentes de Variância das Liquidações de Sinistros de Auto.
NOTE: Option TITLE2 changed to ANOVA de Efeitos Aleatórios: Região / Filial / Regulador.
NOTE: PROC NESTED nested ANOVA / variance components

NOTE: PROC NESTED statement used.


## Etapa 4 — Analisar duas respostas ao mesmo tempo

A administração também se preocupa com o **tempo de ciclo** — quanto
tempo os sinistros levam para serem liquidados. Adicionamos `CycleDays`
à lista `VAR`. Com mais de uma variável dependente, `PROC NESTED`
adicionalmente reporta uma **análise de covariação** mostrando como as
duas respostas covariam em cada nível da hierarquia (por exemplo, se as
regiões que pagam mais também liquidam mais devagar).


In [4]:
TÍTULO 'Componentes de Variância de Liquidação e Tempo de Ciclo';
title2 'Duas Respostas na Mesma Hierarquia Aninhada';

PROCEDIMENTO nested DADOS=claims;
   CLASSE Region Branch Adjuster;
   VARIÁVEL Settlement CycleDays;
   RÓTULO Region='Região' Branch='Filial' Adjuster='Regulador de Sinistros'
         Settlement = 'Indenização Paga (USD)'
         CycleDays  = 'Dias até a Liquidação';
EXECUTAR;


                                Componentes de Variância de Liquidação e Tempo de Ciclo                                 
                                      Duas Respostas na Mesma Hierarquia Aninhada                                       


                       Nested Random Effects Analysis of Variance

Nested Random Effects Analysis of Variance for Variable Indenização Paga (USD)
Variance Source       DF  Sum of Squares       F Value      Pr > F  Error Term   Mean Square  Variance Component    Percent of Total    EMS Coef
Região                 4  62114122.126866          6.71      0.0304    Filial  15528530.531717  1651824.844989             54.4057      8.0000
Filial                 5  11569658.859028          1.89      0.1829  Regulador de Sinistros  2313931.771806   272682.828942              8.9813      4.0000
Regulador de Sinistros  10  12232004.560376          1.22      0.3348     Error  1223200.456038   111582.782346              3.6752      2.0000
Error                 


NOTE: Option TITLE changed to Componentes de Variância de Liquidação e Tempo de Ciclo.
NOTE: Option TITLE2 changed to Duas Respostas na Mesma Hierarquia Aninhada.
NOTE: PROC NESTED nested ANOVA / variance components

NOTE: PROC NESTED statement used.


## Etapa 5 — Visão somente de variância com a opção AOV

Para um resumo rápido dos componentes de variância nas duas respostas, a
opção `AOV` restringe a saída às estatísticas de análise de variância e
**suprime a seção de análise de covariação**. Esta é a visão compacta que
um analista de portfólio examinaria quando precisa apenas do
detalhamento de variância por nível para cada resposta, não da
covariação entre respostas.


In [5]:
TÍTULO 'Somente Componentes de Variância (AOV)';

PROCEDIMENTO nested DADOS=claims aov;
   CLASSE Region Branch Adjuster;
   VARIÁVEL Settlement CycleDays;
   RÓTULO Region='Região' Branch='Filial' Adjuster='Regulador de Sinistros'
         Settlement = 'Indenização Paga (USD)'
         CycleDays  = 'Dias até a Liquidação';
EXECUTAR;

TÍTULO;


                                         Somente Componentes de Variância (AOV)                                         
                                      Duas Respostas na Mesma Hierarquia Aninhada                                       


                       Nested Random Effects Analysis of Variance

Nested Random Effects Analysis of Variance for Variable Indenização Paga (USD)
Variance Source       DF  Sum of Squares       F Value      Pr > F  Error Term   Mean Square  Variance Component    Percent of Total    EMS Coef
Região                 4  62114122.126866          6.71      0.0304    Filial  15528530.531717  1651824.844989             54.4057      8.0000
Filial                 5  11569658.859028          1.89      0.1829  Regulador de Sinistros  2313931.771806   272682.828942              8.9813      4.0000
Regulador de Sinistros  10  12232004.560376          1.22      0.3348     Error  1223200.456038   111582.782346              3.6752      2.0000
Error                 


NOTE: Option TITLE changed to Somente Componentes de Variância (AOV).
NOTE: PROC NESTED nested ANOVA / variance components

NOTE: PROC NESTED statement used.


## Interpretando os resultados

A coluna **Percentual do Total** na tabela de análise de variância é a
informação principal. Lendo-a de cima para baixo, obtemos a parcela da
variabilidade total da liquidação atribuível a cada camada organizacional.
Para o valor da liquidação, a execução produz:

- **Região — 54,4%** (Pr > F = 0,0304). Os dados foram gerados com a
  maior dispersão no nível de região, e a análise a recupera: mais da
  metade de toda a variabilidade da liquidação está *entre* regiões,
  evidência estatisticamente significativa de que *onde* um sinistro é
  tratado é o fator dominante.
- **Filial (dentro da Região) — 9,0%** (Pr > F = 0,18). Uma fatia
  adicional modesta ao descer da região para a filial individual; não
  significativa aqui.
- **Regulador (dentro da Filial) — 3,7%** (Pr > F = 0,33). Pouca
  variação é rastreável ao regulador individual nesta carteira de
  negócios.
- **Erro — 32,9%.** O ruído residual de sinistro para sinistro dentro de
  um regulador; esta é a variação irredutível que nenhuma alavanca
  organizacional pode remover.

Cada nível também carrega um **teste F (Pr > F)** da hipótese nula de que
seu componente de variância é zero. O pequeno valor-p da Região (0,0304)
é evidência estatística de diferenças sistemáticas genuínas entre
regiões, não acaso amostral.

A resposta de tempo de ciclo conta uma história paralela: **a Região
responde por 45,8%** da variação nos dias até a liquidação (Pr > F =
0,0448, novamente significativo), com Filial e Regulador contribuindo com
fatias de um dígito e o resíduo carregando 40,1%. Assim, tanto *quanto*
uma seguradora paga quanto *quanto tempo* leva são governados primeiro e
principalmente pela região.

A execução com duas respostas adiciona uma **análise de covariação**.
Aqui o nível de Região impulsiona os produtos cruzados, e o **coeficiente
de correlação geral é -0,36** — uma relação negativa: as regiões que
pagam liquidações maiores tendem a *encerrá-las mais rápido*, não mais
devagar. Esse é um achado útil e não óbvio — as regiões mais caras não
são as mais lentas.

**Conclusão para o negócio:** como a Região domina o detalhamento
percentual do total para ambas as respostas, a seguradora deve
padronizar as diretrizes de liquidação e os limites de alçada *entre as
regiões* primeiro — é onde vive a maior inconsistência estatisticamente
significativa — antes de investir em intervenções no nível de filial ou
regulador. A correlação negativa entre liquidação e tempo de ciclo
significa que não há uma única região que seja ao mesmo tempo a mais
cara e a mais lenta, então os dois problemas podem ser atacados com
programas separados, direcionados por região. O PROC NESTED transforma
uma sensação vaga de "nossas liquidações são inconsistentes" em uma
atribuição acionável, camada por camada, de onde essa inconsistência
reside.
